# 🎬 ROI e Avaliação de Filmes — TMDB 5000

**Pergunta de negócio:** quais gêneros de filme dão o maior Retorno sobre Investimento (ROI)
e quais fatores (duração, ano, orçamento) estão mais correlacionados com a nota do público?

$$ROI = \frac{revenue - budget}{budget}$$

Dataset: [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) (Kaggle).


## 1. Carregamento dos dados

Antes de qualquer análise, é preciso entender o formato bruto: quantas linhas e colunas
temos, e como estão os tipos de dados.

In [ ]:
import ast
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA_PATH = Path('../data/tmdb_5000_movies.csv')

movies_raw_df = pd.read_csv(DATA_PATH, sep=',')
print(movies_raw_df.shape)
movies_raw_df.head()

## 2. Limpeza

Trabalhamos sempre a partir de `movies_raw_df`, nunca sobrepondo a própria variável —
isso garante que a célula seja reprodutível, independente de quantas vezes for executada.

Removemos:
- linhas sem `release_date`, `budget` ou `revenue`
- filmes com `budget` ou `revenue` menores ou iguais a zero, já que inviabilizam o
  cálculo de ROI (divisão por zero ou números sem sentido de negócio — provavelmente
  dados não preenchidos no TMDB, não filmes de orçamento realmente nulo).

In [ ]:
movies_clean_df = (
    movies_raw_df
    .drop(columns=['homepage', 'overview', 'tagline'])
    .dropna(subset=['release_date', 'budget', 'revenue'])
    .copy()
)

n_antes = len(movies_clean_df)
movies_clean_df = movies_clean_df.query('budget > 0 and revenue > 0')
print(f'Removidas {n_antes - len(movies_clean_df)} linhas com budget/revenue <= 0')

movies_clean_df['release_date'] = pd.to_datetime(movies_clean_df['release_date'], errors='coerce')
movies_clean_df['release_year'] = movies_clean_df['release_date'].dt.year

movies_clean_df.info()

## 3. Feature engineering

Criamos as colunas derivadas necessárias para responder à pergunta de negócio:
`profit`, `roi` e o gênero principal de cada filme (a coluna `genres` original vem
como uma string de lista de dicionários, então extraímos só o primeiro gênero listado).

In [ ]:
movies_feat_df = movies_clean_df.copy()

movies_feat_df['profit'] = movies_feat_df['revenue'] - movies_feat_df['budget']
movies_feat_df['roi'] = (movies_feat_df['revenue'] - movies_feat_df['budget']) / movies_feat_df['budget']


def main_primary_extract(genres_str: str) -> str:
    """Extrai o nome do primeiro gênero da coluna 'genres' (string de lista de dicts)."""
    if pd.isna(genres_str):
        return 'Other'
    try:
        generos = ast.literal_eval(genres_str)
    except (ValueError, SyntaxError):
        return 'Other'
    if not generos:
        return 'Other'
    return generos[0]['name']


movies_feat_df['main_genre'] = movies_feat_df['genres'].apply(main_primary_extract)

movies_feat_df['main_genre'].value_counts()

**Observação:** o dataset é dominado por Drama, Comedy e Action — isso é relevante
na hora de interpretar os resultados por gênero: categorias com poucos filmes (ex:
Foreign, TV Movie) tendem a ter ROI mais instável, já que um único outlier pesa muito
mais na média/mediana.

## 4. Análise

### 4.1 ROI médio por gênero

In [ ]:
roi_by_genre = (
    movies_feat_df
    .groupby('main_genre')['roi']
    .agg(avg_roi='mean', median_roi='median', movies_qnt='count')
    .sort_values('avg_roi', ascending=False)
)

roi_by_genre

> **Nota:** `avg_roi` é bastante sensível a outliers — um único filme de baixíssimo
> orçamento e alta bilheteria pode inflar a média de um gênero inteiro.
> `median_roi` costuma ser uma leitura mais confiável do "gênero típico", e é a métrica
> usada na visualização final desta análise.

### 4.2 Correlação com a nota do público (`vote_average`)

In [ ]:
numerics_columns = ['runtime', 'release_year', 'budget', 'revenue', 'roi', 'vote_average']

score_correlation = (
    movies_feat_df[numerics_columns]
    .corr(numeric_only=True)['vote_average']
    .drop('vote_average')
    .sort_values(key=abs, ascending=False)
)
score_correlation

Vale reforçar: correlação não é causalidade. Se `runtime` aparecer correlacionado
com `vote_average`, isso não significa que aumentar a duração de um filme melhora a nota —
pode ser que filmes mais longos tendam a ser de gêneros já mais bem avaliados (dramas,
por exemplo), e essa seria uma hipótese para investigar em uma próxima etapa.

## 5. Gráficos

### ROI mediano por gênero

In [ ]:
plot_df = (
    roi_by_genre
    .sort_values("median_roi")
    .reset_index()
)

fig_roi = px.bar(
    plot_df,
    x="median_roi",
    y="main_genre",
    orientation="h",
    text="median_roi",
    color="median_roi",
    color_continuous_scale="Blues"
)

fig_roi.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig_roi.update_layout(
    title="ROI Mediano por Gênero",
    xaxis_title="ROI Mediano",
    yaxis_title="Gênero",
    template="simple_white",
    showlegend=False
)

fig_roi.show()

### Nota média por gênero

In [ ]:
vote_by_genre = (
    movies_feat_df
    .groupby('main_genre')['vote_average']
    .agg(avg_vote='mean', median_vote='median', desvio='std', movies_qnt='count')
    .sort_values('avg_vote')
)

plot_vote_df = (
    vote_by_genre
    .sort_values("avg_vote")
    .reset_index()
)

fig_vote = px.bar(
    plot_vote_df,
    x="avg_vote",
    y="main_genre",
    orientation="h",
    text="avg_vote",
    color="avg_vote",
    color_continuous_scale="Blues"
)

fig_vote.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig_vote.update_layout(
    title="Média de Votos por Gênero",
    xaxis_title="Média de votos",
    yaxis_title="Gênero",
    template="simple_white",
    showlegend=False
)

fig_vote.show()